 Feel free to add or change anything; this is based on my step1/2 solutions.

 

In [1]:
import pandas as pd

nhanes = pd.read_csv("../datasets/nhanes.csv")

## Preprocessing Pipeline

In [2]:
#Processing for targets


def bpSYmap(bp):
    if bp >= 120:
        return 1
    else:
        return 0
def bpDImap(bp):
    if bp >= 80:
        return 1
    else:
        return 0 

def normmap(bp):
    if bp == 0:
        return "normal"
    else:
        return "abnormal"

def cholmap(chol):
    if chol < 200:
        return "normal"
    elif chol <= 239:
        return "borderline"
    else:
        return "high"


#sum the two maps; now we have nonzero values where systolic or diasystolic is high
nhanes["BP"] = nhanes["BPXOSY1"].map(bpSYmap) + nhanes["BPXODI1"].map(bpDImap)

#apply the text labels
nhanes["BP"] = nhanes["BP"].map(normmap)


nhanes["CHOL"] = nhanes["LBXTC"].map(cholmap)
nhanes.drop(labels=["BPXOSY1", "BPXODI1", "LBXTC"], axis=1, inplace=True)

In [3]:
nhanes

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDRETH1,DMQMILIZ,DMDBORN4,DMDYRUSR,DMDEDUC2,...,HIQ032D,HIQ032F,HIQ032H,HIQ032I,HIQ210,INDFMMPC,INQ300,IND310,BP,CHOL
0,132141.0,12.0,2.0,2.0,56.0,4.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,borderline
1,136977.0,12.0,2.0,1.0,29.0,3.0,2.0,2.0,2.0,5.0,...,4.0,NaN,NaN,NaN,2.0,3.0,2.0,2.0,abnormal,normal
2,134280.0,12.0,2.0,2.0,25.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,normal
3,132522.0,12.0,2.0,2.0,64.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,abnormal,borderline
4,138725.0,12.0,2.0,2.0,28.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,borderline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5698,138255.0,12.0,2.0,2.0,65.0,5.0,2.0,1.0,NaN,4.0,...,NaN,NaN,8.0,NaN,2.0,1.0,2.0,1.0,normal,normal
5699,134584.0,12.0,2.0,1.0,56.0,1.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,NaN,2.0,3.0,2.0,3.0,abnormal,borderline
5700,133659.0,12.0,2.0,2.0,80.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,9.0,2.0,NaN,NaN,NaN,abnormal,normal
5701,131701.0,12.0,2.0,2.0,46.0,3.0,2.0,1.0,NaN,3.0,...,NaN,NaN,NaN,NaN,1.0,2.0,1.0,NaN,abnormal,high


In [4]:
bp_Rock_Doves = nhanes
chol_Rock_Doves = nhanes

from sklearn.model_selection import train_test_split

bp_training_Rock_Doves, bp_test_Rock_Doves = train_test_split(bp_Rock_Doves, train_size=.8, random_state=42, stratify=bp_Rock_Doves["BP"])

chol_training_Rock_Doves, chol_test_Rock_Doves = train_test_split(chol_Rock_Doves, train_size=.8, random_state=42, stratify=chol_Rock_Doves["CHOL"])

In [5]:
#It's possible that this can be reduced to fewer pipelines using metadata routing or more column transformers

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler



#We need to remove placehold values before imputing

#drop SEQN, SDDSRVYR, RIDSTATR



#remove 7, 9: INQ300 INDFMMPC HIQ210 HIQ011 ALQ151 ALQ111 DMDEDUC2 DMQMILIZ


#Pipeline to remove 7/9 placeholder values
def singleremover(dat):
    if dat == 7 or dat == 9:
        return np.nan
    else:
        return dat
    
def singleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(singleremover)
    return df

srtransform = FunctionTransformer(singleremovermap)

#Remove placeholders for ordinal values that do not need to be 1-hotted
srpipe = Pipeline([
    ("remove 7/9", srtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])

#Pipeline including 1-hot encoder for categorical 
srhotpipe = Pipeline([
    ("remove 7/9", srtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))

])

#remove 77, 99: DMDBORN4, IND310, HIQ032A, ALQ121, DMDMARTZ, DMDYRUSR

def doubleeremover(dat):
    if dat == 77 or dat == 99:
        return np.nan
    else:
        return dat
    
def doubleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(doubleeremover)
    return df

drtransform = FunctionTransformer(doubleremovermap)

drpipe = Pipeline([
    ("remove 77/99", drtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])


drhotpipe = Pipeline([
    ("remove 77/99", drtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))
])


#remove 777, 999: ALQ130, ALQ170

def tripleeremover(dat):
    if dat == 77 or dat == 99:
        return np.nan
    else:
        return dat
    
def tripleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(tripleeremover)
    return df

trtransform = FunctionTransformer(tripleremovermap)

trpipe = Pipeline([
    ("remove 777/999", drtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])

trhotpipe = Pipeline([
    ("remove 777/999", drtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))
])


hotpipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(min_frequency=0.01))
])



#special pipeline transform for ALQ121, which has values out of order; "never" is ranked highest

def alc(dat):
    if dat == 77 or dat == 99:
        return np.nan
    if dat == 0:
        return 11       #to put the "never" value in its proper place
    else:
        return dat
    
def alcmap(df):
    for column in df.columns:
        df[column] = df[column].map(alc)
    return df

alctransform = FunctionTransformer(alcmap)

alcpipe = Pipeline([
    ("remove 777/999", alctransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])


#Special transform for RIDAGEYR, which is capped at 80 but has an average value for the capped bucket of 85

def age(dat):
    if dat == 80:
        return 85
    else:
        return dat
    
def agemap(df):
    for column in df.columns:
        df[column] = df[column].map(age)
    return df

agetransform = FunctionTransformer(agemap)

agepipe = Pipeline([
    ("remove 777/999", agetransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])



#pipeline to impute for features that need no other proccessing
imppipe = Pipeline([
    ("impute", SimpleImputer())         #can't remove the 7/9 values here because the values to be removed are different for different columns
])

In [6]:
plineb = ColumnTransformer([
    #drop seqn and SDDSRVYR, RIDSTATR


    ("remove 7/9", srpipe, ["INDFMMPC",  "HIQ011", "ALQ151", "ALQ111", "DMDEDUC2"]),
    ("remove 7/9 and 1hot", srhotpipe, ["DMQMILIZ", "HIQ210", "INQ300"]),
    
    ("remove 77/99", drpipe, ["IND310", "HIQ032A", "ALQ121",  "DMDYRUSR"]),
    ("remove 77/99 and 1hot", drhotpipe, ["DMDBORN4", "DMDMARTZ"]),
    
    ("remove 777/999", trpipe, ["ALQ130", "ALQ170"]),


    ("remove 77/99 and set 0 = 11", alcpipe, ["ALQ121"]),

    #impute all the rest. All transforms are done concurrently so had to impute the above ones seperatesly

    ("impute", imppipe, [ 
       'DMDHHSIZ',
        'ALQ151',
       'HIQ011',  
        #'HIQ032B', 'HIQ032D', 'HIQ032F', 'HIQ032H', 'HIQ032I',
        'INDFMMPC']),

    ("fix age bucket and impute", agepipe, [ 'RIDAGEYR']),
    
    ("onehot", hotpipe, ["RIAGENDR", "RIDRETH1"])



], remainder="drop")    #everything is at least going through the imputer. Would be faster to not imput when we know there are no NaNs?

In [7]:
#unless we decide to do different preprocessing on chol and bp
plinec=plineb

## Model Training

In [8]:
from sklearn.model_selection import RandomizedSearchCV

#perform random search and return best model. Remember to include metadata routing in argument names 
def RandSearch(model, params):
    RS = RandomizedSearchCV(model, params).fit( X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["CHOL"]]))

    print(RS.cv_results_, RS.get_params)
    return RS.best_estimator_

In [9]:
from sklearn.svm import SVC

from sklearn.ensemble import AdaBoostClassifier

from sklearn.linear_model import SGDClassifier




Svc = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("SVC", SVC(random_state=42))
])

AdaBC = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("AdaBoost", AdaBoostClassifier(random_state=42))
])

SgdC = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("SGDClassifier", SGDClassifier(random_state=42))
])

In [ ]:
#example using just a few hyperparameters
bestSGD = RandSearch(SgdC, dict(SGDClassifier__loss = ['hinge', 'log_loss', 'log', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'], SGDClassifier__alpha = [0.0001,0.00015]))

## Model Evaluation

In [ ]:
from sklearn.model_selection import cross_validate

#cross validate model on both datasets and return results as bp, chol
def CV(model):
    print(list(model.named_steps.keys())[-1])
    bp =cross_validate(estimator=model, X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["BP"]]),  cv=10, scoring=["f1_macro", "recall_macro", "accuracy"])

    print("BP",
            bp)

    chol = cross_validate(estimator=model, X=chol_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["CHOL"]]),  cv=10, scoring=["f1_macro", "recall_macro", "accuracy"])
    
    print("CHOL",
        chol)
    return bp, chol

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from scipy.stats import uniform

## Model Submission

In [ ]:
#Not sure what we'll need to do to make this work on the lab computer

import cloudpickle
#  Svc
SgdC.fit( X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["CHOL"]]))

# Dump your fitted model into `model.joblib` within the current directory
with open('model.pkl', 'wb') as f:
    cloudpickle.dump(SgdC, f)
from huggingface_hub import HfApi, login, create_repo

# Ensure you use a token that has 'write' access
login()
api = HfApi()
from submit import upload_files

repo_id = "Project"
upload_files(api, repo_id, display)

ModuleNotFoundError: No module named 'huggingface_hub'